<a href="https://www.kaggle.com/code/augustinekuo/worker-ppe-detection-train?scriptVersionId=351129098" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<a href="https://www.kaggle.com/code/augustinekuo/worker-ppe-detection-train?scriptVersionId=350095850" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# PPE YOLOv8 training (Colab / Kaggle)

Use **this notebook** for E0–E4. Local 8GB GPUs are only for baseline val, export, and `--batch 8` smokes.

Product bars: **vest / no_vest 95%+**, helmets next, goggles ~70% OK. **Boots are not in this cycle.**

1. Runtime → GPU (Colab) or GPU accelerator (Kaggle), with Internet on.
2. Add secret `ROBOFLOW_API_KEY` (Colab userdata / Kaggle Add-ons → Secrets).
3. Run all cells. Default experiment: `e0_n` on the 12k subset after Combined download + remap.
4. The **smoke test** cell runs first (~3 min). If it raises, stop and fix — the long run would have failed the same way.
5. The **sanity check** cell after training compares independent test mAP50 to training-time val mAP50 and aborts on a mismatch.

Prefer **one** of Colab or Kaggle, not both.

In [1]:
import os
from pathlib import Path

# Colab secret, then Kaggle, then env (local fallback).
try:
    from google.colab import userdata
    os.environ.setdefault("ROBOFLOW_API_KEY", userdata.get("ROBOFLOW_API_KEY"))
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ.setdefault("ROBOFLOW_API_KEY", UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
    except Exception:
        pass

assert os.environ.get("ROBOFLOW_API_KEY"), "Set ROBOFLOW_API_KEY as a Colab/Kaggle secret"
print("key_set", "colab" if IN_COLAB else "kaggle_or_local")

REPO = Path("/content/ppe") if IN_COLAB else Path("/kaggle/working/ppe")
if (REPO / "scripts" / "train.py").exists():
    # Repo already checked out from a previous run in this session (Kaggle/Colab
    # keep /kaggle/working and /content between cell reruns) — pull latest instead
    # of silently training against a stale, possibly-already-fixed-upstream copy.
    print(f"{REPO} already exists — pulling latest instead of re-cloning")
    !git -C {REPO} fetch --depth 1 origin main
    !git -C {REPO} reset --hard origin/main
else:
    REPO.mkdir(parents=True, exist_ok=True)
    !git clone --depth 1 https://github.com/A-Kuo/Worker-Safety-PPE-Detection-Model.git {REPO}
os.chdir(REPO)
print("cwd", Path.cwd())
!git -C {REPO} log -1 --oneline

key_set kaggle_or_local
Cloning into '/kaggle/working/ppe'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 152 (delta 2), reused 117 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 49.75 MiB | 27.54 MiB/s, done.
Resolving deltas: 100% (2/2), done.
cwd /kaggle/working/ppe
70e9b00 (grafted, HEAD -> main, origin/main, origin/HEAD) Runtime fixture


In [2]:
# ultralytics is pinned to the version the configs/scripts were validated against
# (get_cfg rejects unknown kwargs, so version drift can break a run mid-setup).
%pip install -q "ultralytics==8.4.37" roboflow pyyaml opencv-python-headless
%pip install -q -e .
import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), "No GPU attached - enable the GPU accelerator before spending session time."

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 41.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ppe (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
cuda True Tesla T4


In [3]:
# Combined (~2.4GB zip) then Hard Hat Universe. Construction is optional for mapped eval.
!python scripts/download_datasets.py --execute --only combined hardhat
!python scripts/remap_labels.py --source data/raw/combined --out data/processed/combined --mapping combined
!python scripts/remap_labels.py --source data/raw/hardhat --out data/processed/hardhat --mapping hhu
!python scripts/make_subset.py --source data/processed/combined --out data/raw/combined_12k --n 12000 --seed 42
!python scripts/analyze_distribution.py

Requesting export roboflow-universe-projects/personal-protective-equipment-combined-model/4/yolov8 -> /kaggle/working/ppe/data/raw/combined
  zip 100.0% [2542363062/2542363062 bytes]
Extracting combined.zip -> /kaggle/working/ppe/data/raw/combined
Downloaded combined to /kaggle/working/ppe/data/raw/combined
Hard Hat Universe: using version 26 (prefer 26 no_nulls_plain)
Requesting export universe-datasets/hard-hat-universe-0dy7t/26/yolov8 -> /kaggle/working/ppe/data/raw/hardhat
  zip 100.0% [245677789/245677789 bytes]
Extracting hardhat.zip -> /kaggle/working/ppe/data/raw/hardhat
Downloaded hardhat to /kaggle/working/ppe/data/raw/hardhat
train: 30765 images, dropped 0 unmapped boxes
valid: 8814 images, dropped 0 unmapped boxes
test: 4423 images, dropped 0 unmapped boxes
Wrote remapped dataset to /kaggle/working/ppe/data/processed/combined
train: 4912 images, dropped 0 unmapped boxes
valid: 1414 images, dropped 0 unmapped boxes
test: 708 images, dropped 0 unmapped boxes
Wrote remapped da

In [4]:
# SMOKE TEST (~3 min on a T4): the same pipeline the long run uses - 1 epoch on 2% of the
# train set, then the same sanity check. It exists to catch plumbing failures (paths, kwargs,
# missing checkpoint, eval/parse errors) BEFORE the multi-hour run. Any exception here aborts
# "Run All" - fix it before spending credits. A "WARNING: ... under-trained" line is expected
# and fine; a failure is not.
import sys
sys.path.insert(0, "scripts")
LOG_DIR = "/content" if IN_COLAB else "/kaggle/working"
!python scripts/train.py --exp e0_n --name smoke --device 0 --batch 16 --epochs 1 --fraction 0.02 > {LOG_DIR}/train_smoke.log 2>&1
!tail -n 15 {LOG_DIR}/train_smoke.log
import sanity_check
sanity_check.run("smoke")

             no_helmet        268        776          0          0          0          0
                  vest        183        368          0          0          0          0
               no_vest         61        121          0          0          0          0
               goggles        226        257          0          0          0          0
            no_goggles        217        267          0          0          0          0
                gloves        144        299          0          0          0          0
             no_gloves        181        390          0          0          0          0
                  mask         92        250          0          0          0          0
               no_mask        106        188          0          0          0          0
                person         59         85          0          0          0          0
                  cone        105        881          0          0          0          0
                ladde

'warn'

In [5]:
# E0 on the 12k subset. Swap --exp: e1_s | e2_focal | e3_augs | e4_full44k
# P100/T4: batch 16. If OOM, add --batch 8. Kaggle GPU sessions cap at ~9h; the 12k subset
# runs (E0-E3) are a few hours, but E4 on the full 44k is close to the cap - watch the clock.
#
# IMPORTANT: redirect to a log file instead of letting output stream into this
# cell. A 100-epoch run's carriage-return-updating progress bars, captured
# verbatim into the notebook's cell-output JSON, can bloat that JSON to the
# point where Kaggle's post-run nbconvert step (which always runs, converting
# the executed notebook to .ipynb/.html for the Output tab) takes HOURS to
# process it - the kernel looks "still running" and keeps billing the whole
# time, even though training itself finished long before. Confirmed exactly
# this happened on 2026-09-15/16: nbconvert's regex-based cell-output
# processing (mistune.py / filter_links.py) took ~2.8 hours on one bloated
# cell alone. Redirecting keeps this cell's own output tiny (just the tail)
# while the full log still lands on disk.
LOG = f"{LOG_DIR}/train_e0_n.log"
!python scripts/train.py --exp e0_n --device 0 --batch 16 > {LOG} 2>&1
print(f"Full log: {LOG}")
!tail -n 60 {LOG}

Full log: /kaggle/working/train_e0_n.log

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     89/100       3.2G        1.2      0.814       1.14         20        640: 100% ━━━━━━━━━━━━ 525/525 5.3it/s 1:40
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 76/76 5.9it/s 12.9s
                   all       2404       6966       0.74      0.743      0.748      0.461

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     90/100       3.2G      1.198       0.81      1.138         14        640: 100% ━━━━━━━━━━━━ 525/525 5.3it/s 1:40
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 76/76 5.9it/s 12.9s
                   all       2404       6966      0.745      0.741      0.749      0.463
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, me

In [6]:
# Sanity-check the just-trained checkpoint on the held-out test split RIGHT NOW, while
# GPU/credits are still available. Logic lives in scripts/sanity_check.py (unit-tested):
# it compares the independent test mAP50 against the run's own training-time val mAP50.
# A large gap means training and eval used different labels (seen twice: an unremapped
# data path in configs/data/combined.yaml, then make_subset.py resolving symlinks back to
# raw/). It raises AssertionError on a mismatch - do NOT copy the checkpoint off the VM then.
import sanity_check
sanity_check.run("e0_n")

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 93% ━━━━━━━━━━━─ 257/277 9.7it/s 26.7s<2.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 94% ━━━━━━━━━━━─ 259/277 9.8it/s 26.9s<1.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 94% ━━━━━━━━━━━─ 260/277 9.6it/s 27.0s<1.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 94% ━━━━━━━━━━━─ 261/277 9.6it/s 27.1s<1.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 95% ━━━━━━━━━━━─ 262/277 9.7it/s 27.2s<1.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 95% ━━━━━━━━━━━─ 263/277 9.6it/s 27.3s<1.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 95% ━━━━━━━━━━━─ 264/277 9.6it/s 27.4s<1.3s
                 Class     Images  Instances      Box(P       

'pass'

The sanity-check cell above already confirms mAP50 looks real before you leave the VM. If it passed: copy `runs/train/e0_n/weights/best.pt` off the VM (Drive / Kaggle output), then run `scripts/calibrate.py` locally and lead the report with **vest / no_vest**.